# DeiT/Swin Quantized ImageNet Workflow

Use this notebook as a controlled workspace: set model/config knobs, load the model, calibrate, optimize, evaluate, and save results in separate steps.

## 1. Environment

In [ ]:
try:
    import torch
    import datasets
    import torchvision
    import tqdm
    from PIL import Image
except ImportError:
    %pip install torch torchvision transformers datasets tqdm pillow


In [ ]:
import copy
import importlib.util
import json
import os
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()
while PROJECT_DIR != PROJECT_DIR.parent and not (PROJECT_DIR / "mrcp_quant").is_dir():
    PROJECT_DIR = PROJECT_DIR.parent

if not (PROJECT_DIR / "mrcp_quant").is_dir():
    PROJECT_DIR = Path("/content/mrcp-tr-ptq")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

runner_path = PROJECT_DIR / "notebooks" / "run_vision_quant.py"
spec = importlib.util.spec_from_file_location("vision_runner", runner_path)
vision_runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vision_runner)

from mrcp_quant import (
    apply_experiment_config,
    apply_layer_quant_overrides,
    get_vision_model_loader,
    load_experiment_config,
    resolve_q_module_list,
    save_experiment_result,
)

print("Project:", PROJECT_DIR)
print("Runner:", runner_path)


## 2. Controls

In [ ]:
# Edit this JSON file for model_name, q_module_list, bit widths, dataset, and sample counts.
CONFIG_PATH = PROJECT_DIR / "notebooks" / "configs" / "quant_config_vision.json"

# Notebook execution controls only.
RUN_SMOKE_FORWARD = False
RUN_CALIBRATION = True
RUN_SCALE_OPTIMIZATION = None  # None means: use scale_optimization.num_samples from JSON.
RUN_EVALUATION = True
SAVE_RESULTS = True
PRINT_MODULE_LIMIT = 40


In [ ]:
config = load_experiment_config(CONFIG_PATH)
print("Loaded config:", CONFIG_PATH)
print(json.dumps(config, indent=2))


## 3. Load Model

In [ ]:
apply_experiment_config(config)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
preprocess = vision_runner.build_preprocess(config)
q_module_list = resolve_q_module_list(config.get("q_module_list", ["QLayerNorm"]))
resolved_q_module_names = vision_runner.q_module_names(q_module_list)

loader = get_vision_model_loader(config["model_name"])
model = loader(
    pretrained=config.get("pretrained", True),
    q_module_list=q_module_list,
    quant=False,
)

applied_layer_quant_overrides = apply_layer_quant_overrides(model, config)
model.to(device)
model.set_q_module_list(q_module_list)
model.set_quant()
model.eval()

print("Device:", device)
print("Model:", type(model).__name__)
print("Quant modules:", resolved_q_module_names)
print("Layer overrides:", applied_layer_quant_overrides or "none")


In [ ]:
paths = vision_runner.quantized_module_paths(model)
print("Quantized module count:", len(paths))
for item in paths[:PRINT_MODULE_LIMIT]:
    print(f"{item['path']}: {item['type']}")
if len(paths) > PRINT_MODULE_LIMIT:
    print(f"... {len(paths) - PRINT_MODULE_LIMIT} more")


## 4. Optional Smoke Forward

In [ ]:
if RUN_SMOKE_FORWARD:
    image_size = config.get("evaluation", {}).get("image_size", 224)
    with torch.no_grad():
        dummy = torch.randn(1, 3, image_size, image_size, device=device)
        logits = model(dummy)
    print("Logits shape:", tuple(logits.shape))
else:
    print("Smoke forward skipped. Set RUN_SMOKE_FORWARD = True to run it.")


## 5. Calibrate

In [ ]:
if RUN_CALIBRATION:
    vision_runner.calibrate_model(model, q_module_list, config, preprocess, device)
else:
    print("Calibration skipped.")


## 6. Optimize Scale Factors

In [ ]:
run_scale_optimization = RUN_SCALE_OPTIMIZATION
if run_scale_optimization is None:
    run_scale_optimization = config.get("scale_optimization", {}).get("num_samples", 0) > 0

if run_scale_optimization:
    vision_runner.optimize_scale_factors(model, q_module_list, config, preprocess, device)
else:
    print("Scale optimization skipped.")


## 7. Evaluate

In [ ]:
if RUN_EVALUATION:
    metrics, average_loss, num_examples = vision_runner.evaluate_model(
        model, config, preprocess, device
    )
    print("Metrics:", metrics)
    print("Loss:", average_loss)
    print("Examples:", num_examples)
else:
    metrics, average_loss, num_examples = {}, None, 0
    print("Evaluation skipped.")


## 8. Save Results

In [ ]:
if SAVE_RESULTS and metrics:
    output_config = copy.deepcopy(config)
    output_config["q_module_list"] = resolved_q_module_names
    result_path = save_experiment_result(
        accuracy=metrics.get("top1"),
        loss=average_loss,
        configuration=output_config,
        quantized=resolved_q_module_names,
        output_dir=PROJECT_DIR / "output",
        extra={
            "task_name": "imagenet",
            "model_name": config["model_name"],
            "quantized_module_paths": vision_runner.quantized_module_paths(model),
            "metrics": metrics,
            "primary_metric_name": "top1",
            "primary_metric_value": metrics.get("top1"),
            "num_val_examples": num_examples,
        },
    )
    print("Saved results:", result_path)
elif not SAVE_RESULTS:
    print("Result saving disabled.")
else:
    print("No metrics to save.")
